# Building Your First AI Agent in Microsoft Foundry

In this notebook, you will learn how to build a simple AI agent from scratch using Microsoft Foundry's Agent Service. We will create a **Friendly Fitness Coach** that can help users with workout plans, exercise tips, and healthy habits.

By the end of this tutorial, you will understand how to:
- Connect to a Foundry project
- Create an AI agent with custom instructions
- Have a live conversation with your agent

## Step 1: Install Required Packages

Before we begin, we need to install the Python libraries that let us talk to Microsoft Foundry. Run the cell below to install everything you need.

In [ ]:
# This installs all the packages we need:
# - azure-ai-projects: lets us work with Microsoft Foundry agents
# - openai: provides the chat interface for talking to agents
# - python-dotenv: loads our secret settings from a .env file
# - azure-identity: handles authentication with Azure
%pip install azure-ai-projects==2.0.0b3 openai==1.109.1 python-dotenv azure-identity

## Step 2: Connect to Your Foundry Project

Here we load our configuration from a `.env` file and import the libraries we need. The `.env` file contains your Foundry project endpoint (the URL of your project) and the name of the AI model you want to use.

In [ ]:
# Import the built-in module for reading environment variables
import os

# Import the function that reads settings from a .env file
from dotenv import load_dotenv

# Import the Azure authentication tool (uses your logged-in Azure account)
from azure.identity import DefaultAzureCredential

# Import the Foundry project client (our main connection to Foundry)
from azure.ai.projects import AIProjectClient

# Load all the key-value pairs from the .env file into environment variables
load_dotenv()

# Read the Foundry project URL from the environment
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

# Read the name of the AI model we want our agent to use
my_model = os.getenv("MODEL_DEPLOYMENT_NAME")

Now we create a **Foundry client**. Think of this as opening a connection to your Foundry project in the cloud. We pass in our endpoint URL and our Azure credentials so Foundry knows who we are.

In [ ]:
# Create the Foundry client — this is our main gateway to the Foundry platform
# DefaultAzureCredential() automatically uses your Azure CLI login or managed identity
foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=DefaultAzureCredential(),
)

## Step 3: Create Our Fitness Coach Agent

Now for the fun part! We define our agent. An agent needs three things:
1. **A name** — so we can refer to it later
2. **A model** — the AI brain powering the agent
3. **Instructions** — a description of how the agent should behave

We are creating a friendly fitness coach that gives workout advice and encouragement.

In [ ]:
# Import the class that defines what kind of agent we want to build
from azure.ai.projects.models import PromptAgentDefinition

# Give our agent a unique name (this is how Foundry identifies it)
coach_agent_name = "fitness-coach"

# Create a new version of the agent in Foundry
# - model: which AI model to use for generating responses
# - instructions: tells the agent how to behave in conversations
coach_agent = foundry_client.agents.create_version(
    agent_name=coach_agent_name,
    definition=PromptAgentDefinition(
        model=my_model,
        instructions="You are a friendly and motivating fitness coach. You help people create workout plans, give exercise tips, and encourage healthy habits. Always be supportive and positive.",
    ),
)

# Print out confirmation that the agent was created successfully
print(f"Agent created (id: {coach_agent.id}, name: {coach_agent.name}, version: {coach_agent.version})")

## Step 4: Start a Conversation

Before we can chat with our agent, we need two things:
1. **A chat client** — this is an OpenAI-compatible client that handles sending and receiving messages
2. **A conversation (chat session)** — this keeps track of the message history so the agent remembers what you said

In [ ]:
# Get an OpenAI-compatible chat client from our Foundry connection
# This client knows how to send messages and receive replies
chat_client = foundry_client.get_openai_client()

# Start a brand-new conversation (like opening a fresh chat window)
# The conversation stores the full history so the agent has context
chat_session = chat_client.conversations.create()

# Show the conversation ID so we know it was created
print(f"Created conversation with id: {chat_session.id}")

## Step 5: Chat with Your Agent

Now we can have a back-and-forth conversation with our Fitness Coach! The loop below will:
1. Ask you to type a message
2. Send it to the agent
3. Print the agent's reply
4. Repeat until you type "exit" or "quit"

Try asking things like:
- "Can you suggest a 20-minute workout for beginners?"
- "How many times a week should I exercise?"
- "What are some healthy post-workout snacks?"

In [ ]:
# This flag keeps the chat loop running — set it to False to stop
is_chatting = True

while is_chatting:
    # Wait for the user to type a message
    user_message = input("You: ")

    # If the user types "exit" or "quit", end the conversation
    if user_message.lower() in ["exit", "quit"]:
        is_chatting = False
        print("Ending the conversation. Stay active and healthy!")
    else:
        # Send the user's message to the Fitness Coach agent and get a reply
        # - conversation: links this message to our ongoing chat session
        # - extra_body: tells Foundry which agent should respond
        # - input: the actual message text from the user
        coach_reply = chat_client.responses.create(
            conversation=chat_session.id,
            extra_body={
                "agent": {
                    "name": coach_agent_name,
                    "type": "agent_reference"
                }
            },
            input=user_message
        )

        # Display the agent's response
        print(f"Coach: {coach_reply.output_text}")